In [ ]:
import pandas as pd
df = pd.read_csv("..\\data\\student-course-completion-prediction-dataset-trimmed.csv")

print(df.shape)
print(df.head())

print(df.dtypes.to_frame(name="dtype"))

In [ ]:
# Keep an untouched copy of the loaded and named dataset
df_copy = df.copy(deep=True)

In [ ]:
# Check missing values
missing = df_copy.isna().sum()
print("Columns containing missing values:")
print(missing[missing > 0].sort_values(ascending=False))

# Check duplicate rows
print("\nNumber of duplicate rows:")
print(df_copy.duplicated().sum())

# Check the ranges of important measurements
range_cols = [
    "Age", 
    "Course_Duration_Days", 
    "Average_Session_Duration_Min", 
    "Video_Completion_Rate", 
    "Time_Spent_Hours",
    "Days_Since_Last_Login",
    "Assignments_Submitted",
    "Assignments_Missed",
    "Quiz_Attempts",
    "Quiz_Score_Avg", 
    "Project_Grade", 
    "Progress_Percentage" 
]

print("\nMinimum and maximum values:")
print(df_copy[range_cols].agg(["min", "max"]))

In [ ]:
review_cols = [
    "Age", # improbable to have students younger than 10 or older than 90
    "Course_Duration_Days", # 0 <= result <= 90
    "Average_Session_Duration_Min", # 0 <= result <= 80
    "Video_Completion_Rate", # 0 <= result <= 100
    "Time_Spent_Hours", # 0 <= result <= 30
    "Days_Since_Last_Login", # 0 <= result <= 100
    "Assignments_Submitted", # submitted + missed assignments total should not exceed 10
    "Assignments_Missed", # submitted + missed assignments total should not exceed 10 
    "Quiz_Attempts", # 0 <= result <= 16
    "Quiz_Score_Avg", # 0 <= result <= 100
    "Project_Grade", # 0 <= result <= 100
    "Progress_Percentage" # 0 <= result <= 100
]

records_to_review = df_copy.loc[
    ((df_copy["Age"] < 10) | (df_copy["Age"] > 90)) |
    ((df_copy["Course_Duration_Days"] < 0) | (df_copy["Course_Duration_Days"] > 90)) |
    ((df_copy["Average_Session_Duration_Min"] < 0) | (df_copy["Average_Session_Duration_Min"] > 80)) |
    ((df_copy["Video_Completion_Rate"] < 0) | (df_copy["Video_Completion_Rate"] > 100)) |
    ((df_copy["Time_Spent_Hours"] < 0) | (df_copy["Time_Spent_Hours"] > 30)) |
    ((df_copy["Days_Since_Last_Login"] < 0) | (df_copy["Days_Since_Last_Login"] > 100)) |
    ((df_copy["Assignments_Submitted"] < 0) | (df_copy["Assignments_Submitted"] > 10)) |
    ((df_copy["Assignments_Missed"] < 0) | (df_copy["Assignments_Missed"] > 10)) |
    ((df_copy["Quiz_Attempts"] < 0) | (df_copy["Quiz_Attempts"] > 16)) |
    ((df_copy["Quiz_Score_Avg"] < 0) | (df_copy ["Quiz_Score_Avg"] > 100)) |
    ((df_copy["Project_Grade"] < 0) | (df_copy["Project_Grade"] > 100)) |
    ((df_copy["Progress_Percentage"] < 0) | (df_copy["Progress_Percentage"] > 100))
].sort_values(
    ["Age", "Course_Duration_Days", "Average_Session_Duration_Min"]
)

print(records_to_review.to_string())

In [ ]:
# Clean a copy of the dataset without changing the original
def clean_data(data):
    # Create a copy so the original data stays unchanged
    clean = data.copy(deep=True)

    # Correct the confirmed age error for STU100001
    # The original dataset shows age 17, not 170
    clean.loc[
        clean["Student_ID"] == "STU100001",
        "Age"
    ] = 17

    for col in [
        "Quiz_Score_Avg",
        "Project_Grade",
        "Progress_Percentage",
        "App_Usage_Percentage"
    ]:
        clean[col] = clean[col].fillna(clean[col].median())

    clean = clean.drop_duplicates()
    clean = clean.reset_index(drop=True)

    return clean

In [ ]:
# Apply the cleaning function to the untouched dataset
df_clean_preview = clean_data(df_copy)

In [ ]:
# Check what type of object records_to_review is
# This helps us understand why it cannot be used directly as a row filter
print(type(records_to_review))

# Show its size
print(records_to_review.shape)

In [ ]:
# Display the record that was flagged for review
# We want to confirm exactly which student is inside records_to_review
records_to_review[
    ["Student_ID", "Name", "Age"]
]

In [ ]:
print("Original shape:", df_copy.shape)
print("Cleaned shape:", df_clean_preview.shape)

print(
    "Rows removed:",
    len(df_copy) - len(df_clean_preview)
)

In [ ]:
range_cols = [
    "Age", 
    "Course_Duration_Days", 
    "Average_Session_Duration_Min", 
    "Video_Completion_Rate", 
    "Time_Spent_Hours",
    "Days_Since_Last_Login",
    "Assignments_Submitted",
    "Assignments_Missed",
    "Quiz_Attempts",
    "Quiz_Score_Avg", 
    "Project_Grade", 
    "Progress_Percentage" 
]

print("\nMinimum and maximum values:")
print(df_copy[range_cols].agg(["min", "max"]))

In [ ]:
invalid_record = (
        ((df_clean_preview["Age"] < 10) | (df_clean_preview["Age"] > 90)) |
        ((df_clean_preview["Course_Duration_Days"] < 0) | (df_clean_preview["Course_Duration_Days"] > 90)) |
        ((df_clean_preview["Average_Session_Duration_Min"] < 0) | (df_clean_preview["Average_Session_Duration_Min"] > 80)) |
        ((df_clean_preview["Video_Completion_Rate"] < 0) | (df_clean_preview["Video_Completion_Rate"] > 100)) |
        ((df_clean_preview["Time_Spent_Hours"] < 0) | (df_clean_preview["Time_Spent_Hours"] > 30)) |
        ((df_clean_preview["Days_Since_Last_Login"] < 0) | (df_clean_preview["Days_Since_Last_Login"] > 100)) |
        ((df_clean_preview["Assignments_Submitted"] < 0) | (df_clean_preview["Assignments_Submitted"] > 10)) |
        ((df_clean_preview["Assignments_Missed"] < 0) | (df_clean_preview["Assignments_Missed"] > 10)) |
        ((df_clean_preview["Quiz_Attempts"] < 0) | (df_clean_preview["Quiz_Attempts"] > 16)) |
        ((df_clean_preview["Quiz_Score_Avg"] < 0) | (df_clean_preview["Quiz_Score_Avg"] > 100)) |
        ((df_clean_preview["Project_Grade"] < 0) | (df_clean_preview["Project_Grade"] > 100)) |
        ((df_clean_preview["Progress_Percentage"] < 0) | (df_clean_preview["Progress_Percentage"] > 100))
    )

print(df_clean_preview.loc[invalid_record].to_string())

In [ ]:
missing = df_clean_preview.isna().sum()
print("\nColumns still containing missing values:")
print(
    missing[missing > 0]
    .sort_values(ascending=False)
)
print(
    "\nDuplicate rows:",
    df_clean_preview.duplicated().sum()
)

In [ ]:
print("\nFirst five cleaned records:")
print(
    df_clean_preview[
        [
            "Student_ID",
            "Name",
            "Gender",
            "Age",
            "Education_Level",
            "Employment_Status",
            "Completed"
        ]
    ].head()
)

In [ ]:
import sqlite3
# Use the cleaned data already reviewed above
df_etl_clean = df_clean_preview.copy(deep=True)

with sqlite3.connect("..\\data\\etl.db") as conn:
    df_etl_clean.to_sql(
        "clean",
        conn,
        if_exists="replace",
        index=False
    )

print("ETL completed.")
print("Cleaned table shape:", df_etl_clean.shape)

In [ ]:
with sqlite3.connect("..\\data\\elt.db") as conn:
    # Load the untouched raw data first
    df_copy.to_sql(
        "raw",
        conn,
        if_exists="replace",
        index=False
    )
    # Read the raw table back into Pandas
    df_elt_raw = pd.read_sql(
        "SELECT * FROM raw",
        conn
    )
    # Transform the data after loading
    df_elt_clean = clean_data(df_elt_raw)
    # Store the cleaned result as a second table
    df_elt_clean.to_sql(
        "clean",
        conn,
        if_exists="replace",
        index=False
    )

print("ELT completed.")
print("Raw table shape:", df_elt_raw.shape)
print("Cleaned table shape:", df_elt_clean.shape)

In [ ]:
with sqlite3.connect("..\\data\\etl.db") as conn:
    etl_tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table'",
        conn
    )
print("Tables in etl.db:")
print(etl_tables)

with sqlite3.connect("..\\data\\elt.db") as conn:
    elt_tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table'",
        conn
    )
print("\nTables in elt.db:")
print(elt_tables)

In [ ]:
print(df_copy.shape)
print(df_copy.dtypes.value_counts())
print(df_copy.info())
print(df_copy.describe())

In [ ]:
import matplotlib.pyplot as plt

plt.hist(df_etl_clean["App_Usage_Percentage"].dropna(), bins=20, edgecolor="white", linewidth=1)
plt.xlabel("Application Usage Percentage"); plt.ylabel("Number of Students")
plt.title("Distribution of Application Usage Percentage")
plt.show()

In [ ]:
plt.hist(df_etl_clean["Age"].dropna(), bins=25, edgecolor="white", linewidth=1)
plt.xlabel("Age")
plt.ylabel("Number of Students")
plt.title("Distribution of Age")
plt.show()

In [ ]:
plt.hist(df_copy["Age"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Age (before cleaning)")
plt.xlabel("Age")
plt.ylabel("Number of Students")
plt.show()
plt.hist(df_copy["App_Usage_Percentage"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Application Usage Percentage (before cleaning)")
plt.xlabel("Application Usage Percentage")
plt.ylabel("Number of Students")
plt.show()
plt.hist(df_copy["Project_Grade"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Project Grade (before cleaning)")
plt.xlabel("Project Grade")
plt.ylabel("Number of Students")
plt.show()

In [ ]:
# Compare the variables that were reviewed during cleaning
for col in ["Age", "Project_Grade", "Quiz_Score_Avg", "Time_Spent_Hours"]:
    print(
        col,
        "raw range:",
        (df_copy[col].min(), df_copy[col].max()),
        "clean range:",
        (df_etl_clean[col].min(), df_etl_clean[col].max())
    )

In [ ]:
plt.hist(df_etl_clean["Age"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Age (after cleaning)")
plt.xlabel("Age")
plt.ylabel("Number of Students")
plt.show()
plt.hist(df_etl_clean["App_Usage_Percentage"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Application Usage Percentage (after cleaning)")
plt.xlabel("Application Usage Percentage")
plt.ylabel("Number of Students")
plt.show()
plt.hist(df_etl_clean["Project_Grade"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Project Grade (after cleaning)")
plt.xlabel("Project Grade")
plt.ylabel("Number of Students")
plt.show()

In [ ]:
plt.boxplot(df_etl_clean["Age"].dropna(), orientation="horizontal", tick_labels=["Age"])
plt.xlabel("Age")
plt.show()

In [ ]:
def iqr_flag(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)

age_flag = iqr_flag(df_etl_clean["age"])
height_flag = iqr_flag(df_etl_clean["height"])
weight_flag = iqr_flag(df_etl_clean["weight"])
demographic_flags = age_flag | height_flag | weight_flag

print(
    df_etl_clean.loc[
        demographic_flags,
        ["age", "sex", "height", "weight", "class"]
    ].sort_values(["age", "height", "weight"]).to_string()
)

# Heart rate is checked separately with a z-score
mean_hr = df_etl_clean["heart_rate"].mean()
std_hr = df_etl_clean["heart_rate"].std()
z_hr = (df_etl_clean["heart_rate"] - mean_hr) / std_hr

print("Heart-rate values with |z| > 3:")
print(df_etl_clean.loc[z_hr.abs() > 3, ["age", "sex", "heart_rate", "class"]])

In [ ]:
print(
    df_etl_clean.loc[
        (df_etl_clean["age"] == 75) &
        (df_etl_clean["sex"] == 0) &
        (df_etl_clean["height"] == 190) &
        (df_etl_clean["weight"] == 80)
    ].to_string()
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].boxplot(df_etl_clean["height"].dropna(), tick_labels=["Height"])
axes[0].set_title("Height"); axes[0].set_ylabel("cm")
axes[1].boxplot(df_etl_clean["heart_rate"].dropna(), tick_labels=["Heart rate"])
axes[1].set_title("Heart rate"); axes[1].set_ylabel("bpm")
axes[2].boxplot(df_etl_clean["weight"].dropna(), tick_labels=["Weight"])
axes[2].set_title("Weight"); axes[2].set_ylabel("kg")
plt.suptitle("Cleaned data before optional capping")
plt.show()

In [ ]:
df_sensitivity = df_etl_clean.copy(deep=True)
df_sensitivity["height_capped"] = df_sensitivity["height"].clip(
    lower=df_sensitivity["height"].quantile(0.01)
)
df_sensitivity["heart_rate_capped"] = df_sensitivity["heart_rate"].clip(
    lower=df_sensitivity["heart_rate"].quantile(0.01),
    upper=df_sensitivity["heart_rate"].quantile(0.99)
)
df_sensitivity["weight_capped"] = df_sensitivity["weight"].clip(
    upper=df_sensitivity["weight"].quantile(0.99)
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].boxplot(df_sensitivity["height_capped"].dropna(), tick_labels=["Height"])
axes[0].set_title("Height (sensitivity-capped)"); axes[0].set_ylabel("cm")
axes[1].boxplot(df_sensitivity["heart_rate_capped"].dropna(), tick_labels=["Heart rate"])
axes[1].set_title("Heart rate (capped)"); axes[1].set_ylabel("bpm")
axes[2].boxplot(df_sensitivity["weight_capped"].dropna(), tick_labels=["Weight"])
axes[2].set_title("Weight (capped)"); axes[2].set_ylabel("kg")
plt.suptitle("Optional sensitivity-only capped view")
plt.show()

In [ ]:
rate = df_etl_clean.groupby("sex", observed=False)["arrhythmia_present"].mean()
print(rate)
sex_labels = {0: "Male", 1: "Female"}
labels = [sex_labels[int(value)] for value in rate.index]
plt.bar(labels, rate.values)
plt.xlabel("Sex")
plt.ylabel("Proportion with arrhythmia")
plt.title("Arrhythmia rate by sex")
plt.show()          # Figure 13 is what you'll see

In [ ]:
import numpy as np

num_cols = ["age", "height", "weight", "qrs_duration", "pr_interval", "qt_interval",
    "heart_rate"]
corr_matrix = df_etl_clean[num_cols].corr()
print(corr_matrix.round(2))

# Find the strongest non-diagonal linear association
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
strongest_pair = upper_triangle.abs().stack().idxmax()
print(
    "Strongest absolute correlation:",
    strongest_pair,
    round(corr_matrix.loc[strongest_pair[0], strongest_pair[1]], 2)
)
plt.imshow(corr_matrix, cmap="RdYlGn", vmin=-1, vmax=1)
plt.xticks(range(len(num_cols)), num_cols, rotation=45, ha="right")
plt.yticks(range(len(num_cols)), num_cols)
plt.colorbar(label="Correlation")
plt.title("Correlation between 7 of 279 input features")
plt.show()          # Figure 14 is what you'll see

In [ ]:
class_names = {
    1: "Normal", 2: "Ischemic changes (CAD)", 3: "Old Ant. MI", 4: "Old Inf. MI",
    5: "Sinus tachycardia", 6: "Sinus bradycardia", 7: "PVC", 8: "Supraventricular PC",
    9: "Left bundle branch block", 10: "Right bundle branch block",
    11: "1st degree AV block", 12: "2nd degree AV block", 13: "3rd degree AV block",
    14: "Left vent. hypertrophy", 15: "Atrial fib./flutter", 16: "Other",
}
pqrst_cols = ["p_interval", "qrs_duration", "pr_interval", "qt_interval", "t_interval",
    "heart_rate"]
class_counts = df_etl_clean["class"].value_counts().sort_index()
print("Patients per class:")
print(class_counts)
profile = df_etl_clean.groupby("class")[pqrst_cols].mean()
print("Mean profile per class:")
print(profile.round(1))
# standardise each column so differing scales (ms vs bpm) do not dominate the color map
profile_z = (profile - profile.mean()) / profile.std()
row_labels = [class_names[int(c)] for c in profile_z.index]
plt.imshow(profile_z.values, cmap="RdYlGn", vmin=-2, vmax=2, aspect="auto")
plt.xticks(range(len(pqrst_cols)), pqrst_cols, rotation=45, ha="right")
plt.yticks(range(len(profile_z)), row_labels)
plt.colorbar(label="Standardised mean (per column)")
plt.title("PQRST + heart rate profile, by arrhythmia type")
plt.show()

In [ ]:
miss = df_copy.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0] * 100
plt.barh(miss.index.astype(str), miss.values)
plt.xlabel("% missing")
plt.title("Missing values in the raw data")
plt.show()

In [ ]:
order = sorted(df_etl_clean["class"].unique())
x_pos = df_etl_clean["class"].map({c: i for i, c in enumerate(order)})
plt.scatter(x_pos, df_etl_clean["heart_rate"], alpha=0.4)
means = df_etl_clean.groupby("class")["heart_rate"].mean()
print("Mean heart rate by class:")
print(means.sort_values())
plt.scatter(range(len(order)), [means[c] for c in order], color="red", marker="D",
    label="Mean per type")
plt.xticks(range(len(order)), [class_names[c] for c in order], rotation=45, ha="right")
plt.xlabel("Arrhythmia type"); plt.ylabel("Heart rate (bpm)")
plt.title("Heart rate by arrhythmia type")
plt.legend()
plt.show()

In [ ]:
from matplotlib.lines import Line2D

colors = df_etl_clean["arrhythmia_present"].map({True: "orange", False: "teal"})
plt.scatter(df_etl_clean["height"], df_etl_clean["weight"], c=colors, alpha=0.5)
plt.xlabel("Height (cm)")
plt.ylabel("Weight (kg)")
plt.title("Height vs weight, by diagnosis")
legend_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="teal", markersize=8, label="No arrhythmia"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="orange", markersize=8, label="Arrhythmia present"),
]
plt.legend(handles=legend_handles)
plt.show()

In [ ]:
jitter = df_etl_clean["arrhythmia_present"].astype(int) + np.random.uniform(-0.08, 0.08,
    len(df_etl_clean))
colors = df_etl_clean["arrhythmia_present"].map({True: "orange", False: "teal"})
plt.scatter(df_etl_clean["age"], jitter, c=colors, alpha=0.5)
plt.yticks([0, 1], ["No", "Yes"])
plt.xlabel("Age")
plt.ylabel("arrhythmia_present")
plt.title("Age vs arrhythmia_present")
plt.show()          # y-axis labels (No/Yes) already identify the two colours here

In [ ]:
colors = df_etl_clean["arrhythmia_present"].map({True: "orange", False: "teal"})
cols = ["height", "weight", "qt_interval", "heart_rate"]
fig, axes = plt.subplots(4, 4, figsize=(9, 9))
for i, c1 in enumerate(cols):
    for j, c2 in enumerate(cols):
        ax = axes[i, j]
        if i == j:
            ax.hist(df_etl_clean[c1].dropna(), bins=20)
        else:
            ax.scatter(df_etl_clean[c2], df_etl_clean[c1], s=4, alpha=0.35, c=colors)
        if i == 3:
            ax.set_xlabel(c2)
        if j == 0:
            ax.set_ylabel(c1)
plt.tight_layout()
from matplotlib.patches import Patch
fig.legend(handles=[
    Patch(color="teal", label="No arrhythmia"),
    Patch(color="orange", label="Arrhythmia present"),
], loc="upper right")
plt.show()

In [ ]:
from scipy import stats

a = df_etl_clean.loc[df_etl_clean["arrhythmia_present"], "heart_rate"]
b = df_etl_clean.loc[~df_etl_clean["arrhythmia_present"], "heart_rate"]
t, p = stats.ttest_ind(a, b, equal_var=False)

print("Mean with arrhythmia:", a.mean())
print("Mean without arrhythmia:", b.mean())

print(f"t = {t:.2f}, p = {p:.4f}")

if p < 0.05:
    print("Reject H0: the mean heart rates differ significantly.")
else:
    print("Fail to reject H0: no significant mean difference was detected.")

In [ ]:
from scipy import stats

table = pd.crosstab(
    df_etl_clean["sex"],
    df_etl_clean["arrhythmia_present"]
)
print(table)
chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"chi2 = {chi2:.2f}, p = {p:.6f}")
if p < 0.05:
    print("Reject H0: sex and diagnosis are associated in this sample.")
else:
    print("Fail to reject H0: no significant association was detected.")

In [ ]:
# Check duplicate Student_ID values
print("Duplicate Student_IDs:", df["Student_ID"].duplicated().sum())

In [ ]:
# Display all rows that share a duplicated Student_ID
# keep=False shows both copies of the duplicated record
df[df["Student_ID"].duplicated(keep=False)]

In [ ]:
# Create a cleaned copy and remove completely identical duplicate rows
df_clean = df.drop_duplicates().copy()

In [ ]:
# Compare the number of rows before and after duplicate removal
print("Before:", df.shape)
print("After:", df_clean.shape)

In [ ]:
# Check that there are no completely duplicated rows remaining
print(
    "Duplicate rows:",
    df_clean.duplicated().sum()
)

# Check that there are no duplicated Student_ID values remaining
print(
    "Duplicate Student_IDs:",
    df_clean["Student_ID"].duplicated().sum()
)

In [ ]:
# Show summary statistics for the Age column
# This helps us inspect the age range before deciding
# whether my teammate's current age-cleaning rule is justified
df_copy["Age"].describe()

In [ ]:
# Show all records where Age is greater than 60
# This lets us inspect the unusual ages before deciding what to clean
df_copy.loc[
    df_copy["Age"] > 60,
    ["Student_ID", "Name", "Age"]
].sort_values("Age")

In [ ]:
# Verify that the confirmed age error was corrected
# Shows STU100001 with Age = 17
df_clean_preview.loc[
    df_clean_preview["Student_ID"] == "STU100001",
    ["Student_ID", "Name", "Age"]
]

In [ ]:
# Check the total number of assignments for each student
# This helps us see whether submitted + missed is consistent across the dataset
assignment_total = (
    df_copy["Assignments_Submitted"]
    + df_copy["Assignments_Missed"]
)

# Show how many students have each total
print(assignment_total.value_counts().sort_index())

In [ ]:
# Compare the total number of assignments across each course
# This helps us check whether some courses genuinely have 9 assignments
# while other courses have 10

assignment_by_course = pd.crosstab(
    df_copy["Course_ID"],
    assignment_total
)

print(assignment_by_course)

In [ ]:
# Show the combinations of assignments submitted and missed
# This helps us see exactly how the total of 9 or 10 is being formed

assignment_combinations = (
    df_copy
    .groupby(["Assignments_Submitted", "Assignments_Missed"])
    .size()
    .reset_index(name="Count")
)

print(assignment_combinations)

NOTE: Assignment totals of 9 and 10 occur across multiple courses and also appear in the original source dataset. Because there is no evidence identifying a missing assignment value, these records were retained unchanged.

In [ ]:
# Find all categorical/text columns in the untouched dataset
# These columns will be checked for inconsistent categories
categorical_cols = df_copy.select_dtypes(
    include=["object", "string"]
).columns

# Display the categorical column names
print(categorical_cols.tolist())

In [ ]:
# Select categorical columns that should contain a limited number of categories
# Student_ID and Name are excluded because they contain many unique values
category_check_cols = [
    "Gender",
    "Education_Level",
    "Employment_Status",
    "City",
    "Device_Type",
    "Internet_Connection_Quality",
    "Course_ID",
    "Course_Name",
    "Category",
    "Course_Level",
    "Payment_Mode",
    "Fee_Paid",
    "Discount_Used",
    "Completed"
]

# Display the unique values in each categorical column
for col in category_check_cols:
    print(f"\n{col}:")
    print(df_copy[col].unique())

In [ ]:
# Check categorical columns for hidden inconsistencies
# Example: "Male", "male", or " Male " would be treated as possible duplicates
for col in category_check_cols:

    # Count categories exactly as they currently appear
    original_count = df_copy[col].nunique(dropna=False)

    # Count categories after removing spaces and ignoring capitalisation
    cleaned_count = (
        df_copy[col]
        .astype("string")
        .str.strip()
        .str.lower()
        .nunique(dropna=False)
    )

    # Only display a result if inconsistent formatting is found
    if original_count != cleaned_count:
        print(
            col,
            "- possible inconsistent categories:",
            original_count,
            "original vs",
            cleaned_count,
            "standardised"
        )

In [ ]:
# List the payment-related columns we want to inspect
payment_cols = [
    "Payment_Mode",
    "Fee_Paid",
    "Payment_Amount",
    "Discount_Used"
]

# Show the values/counts in each payment column
# This helps us understand how the payment data is structured
for col in payment_cols:
    print(f"\n{col}:")
    print(df_copy[col].value_counts(dropna=False))

In [ ]:
# Compare Payment_Mode with Fee_Paid
# This checks whether each payment method has the expected Yes/No payment status
payment_fee_check = pd.crosstab(
    df_copy["Payment_Mode"],
    df_copy["Fee_Paid"]
)

print(payment_fee_check)

In [ ]:
# Check whether each payment mode has a zero or non-zero payment amount
# This helps detect cases such as a Free course having a positive payment amount

payment_amount_check = pd.crosstab(
    df_copy["Payment_Mode"],
    df_copy["Payment_Amount"] == 0
)

print(payment_amount_check)

In [ ]:
# Show summary statistics for Scholarship payment amounts
# This helps us understand what Payment_Amount represents for scholarship students
df_copy.loc[
    df_copy["Payment_Mode"] == "Scholarship",
    "Payment_Amount"
].describe()

In [ ]:
# Compare payment method with whether a discount was used
# This helps us check how discounts are recorded for each payment type
payment_discount_check = pd.crosstab(
    df_copy["Payment_Mode"],
    df_copy["Discount_Used"]
)

print(payment_discount_check)

In [ ]:
# Inspect Free-course records where Discount_Used is Yes
# This checks whether their payment amount is still correctly recorded as 0
free_discount = df_copy[
    (df_copy["Payment_Mode"] == "Free")
    & (df_copy["Discount_Used"] == "Yes")
]

# Show how many such records exist
print("Free courses with discount:", len(free_discount))

# Show the payment amounts for these records
print(free_discount["Payment_Amount"].value_counts())

In [ ]:
# Check the range of Payment_Amount
# Payment amounts should not be negative
print("Minimum payment amount:", df_copy["Payment_Amount"].min())
print("Maximum payment amount:", df_copy["Payment_Amount"].max())

# Count any negative payment amounts
print(
    "Negative payment amounts:",
    (df_copy["Payment_Amount"] < 0).sum()
)

NOTE: Scholarship records show Fee_Paid = No with a positive Payment_Amount. This pattern also appears in the original source dataset, so it was retained. Fee_Paid = No is interpreted as the student not personally paying the fee, while Payment_Amount may represent the amount covered by the scholarship.

In [ ]:
# Check how many different Course_Name values are linked to each Course_ID
course_id_check = (
    df_copy
    .groupby("Course_ID")["Course_Name"]
    .nunique()
    .sort_values(ascending=False)
)

print(course_id_check)

In [ ]:
# Find all numerical columns in the raw dataset
numerical_cols = df_copy.select_dtypes(include="number").columns

# Check the minimum value and number of negative values in each numerical column
for col in numerical_cols:
    minimum_value = df_copy[col].min()
    negative_count = (df_copy[col] < 0).sum()

    print(
        f"{col}: "
        f"minimum = {minimum_value}, "
        f"negative values = {negative_count}"
    )

In [ ]:
# Check the minimum and maximum values for the rating columns
# Both ratings should stay between their valid limits and must not exceed 5

for col in ["Instructor_Rating", "Satisfaction_Rating"]:
    print(
        f"{col}: "
        f"minimum = {df_copy[col].min()}, "
        f"maximum = {df_copy[col].max()}, "
        f"values above 5 = {(df_copy[col] > 5).sum()}"
    )

In [ ]:
# List the columns that represent percentages/rates
percentage_cols = [
    "Video_Completion_Rate",
    "Progress_Percentage",
    "App_Usage_Percentage"
]

# Check the minimum, maximum, and how many values exceed 100
for col in percentage_cols:
    print(
        f"{col}: "
        f"minimum = {df_copy[col].min()}, "
        f"maximum = {df_copy[col].max()}, "
        f"values above 100 = {(df_copy[col] > 100).sum()}"
    )

In [ ]:
# Show which dataframe variables currently exist
print([
    name
    for name in globals()
    if name.startswith("df")
])

In [ ]:
# Confirm df_copy is loaded correctly
print(df_copy.shape)

In [ ]:
# Check the valid responses in binary text columns
binary_cols = [
    "Fee_Paid",
    "Discount_Used",
    "Completed"
]

for col in binary_cols:
    print(f"\n{col}:")
    print(df_copy[col].value_counts(dropna=False))

In [ ]:
# Check whether a Country column exists in the dataset
print("Country column exists:", "Country" in df_copy.columns)

# Show all unique course names
print("\nCourse Names:")
print(sorted(df_copy["Course_Name"].dropna().unique()))

In [ ]:
# Compare every Course_Name with every other Course_Name
# This helps find names that are very similar and may be accidental misspellings

from difflib import SequenceMatcher

course_names = df_copy["Course_Name"].dropna().unique()

for i in range(len(course_names)):
    for j in range(i + 1, len(course_names)):

        # Calculate how similar the two course names are
        similarity = SequenceMatcher(
            None,
            course_names[i].lower(),
            course_names[j].lower()
        ).ratio()

        # Only display pairs that are at least 80% similar
        if similarity >= 0.80:
            print(
                course_names[i],
                "<->",
                course_names[j],
                "similarity =",
                round(similarity, 2)
            )

In [ ]:
# Check the number of missing values in every column
missing_values = df_copy.isna().sum()

# Show only columns that actually contain missing values
print(missing_values[missing_values > 0])

In [ ]:
# Check the data type of every column
print(df_copy.dtypes)

# Specifically inspect Enrollment_Date
print("\nEnrollment_Date data type:")
print(df_copy["Enrollment_Date"].dtype)

# Show a few Enrollment_Date values
print("\nExample Enrollment_Date values:")
print(df_copy["Enrollment_Date"].head())

In [ ]:
# Convert Enrollment_Date temporarily for inspection only
# dayfirst=True because the dataset uses DD/MM/YYYY
enrollment_date_check = pd.to_datetime(
    df_copy["Enrollment_Date"],
    dayfirst=True,
    errors="coerce"
)

# Count dates that could not be converted
print(
    "Invalid Enrollment_Date values:",
    enrollment_date_check.isna().sum()
)

# Show the earliest and latest enrollment dates
print("Earliest enrollment date:", enrollment_date_check.min())
print("Latest enrollment date:", enrollment_date_check.max())

In [ ]:
# Show the minimum and maximum for every numerical column
# This helps identify values that may need further investigation

numerical_ranges = df_copy.select_dtypes(include="number").agg(["min", "max"]).T

print(numerical_ranges)

In [ ]:
# Find records where App_Usage_Percentage is exactly 100
app_usage_100 = df_copy[
    df_copy["App_Usage_Percentage"] == 100
]

# Count how many records have exactly 100%
print("Records with App_Usage_Percentage = 100:", len(app_usage_100))

# Show the student IDs and usage percentage
print(
    app_usage_100[
        ["Student_ID", "App_Usage_Percentage"]
    ].head(20)
)

In [ ]:
# Final audit of every column in the untouched dataset
# This checks:
# - data type
# - missing values
# - number of unique values
# - minimum and maximum values for numerical columns

audit_rows = []

for col in df_copy.columns:

    # Basic information for every column
    column_info = {
        "Column": col,
        "Data_Type": str(df_copy[col].dtype),
        "Missing": df_copy[col].isna().sum(),
        "Unique": df_copy[col].nunique(dropna=False)
    }

    # Add minimum and maximum only if the column is numerical
    if pd.api.types.is_numeric_dtype(df_copy[col]):
        column_info["Min"] = df_copy[col].min()
        column_info["Max"] = df_copy[col].max()
    else:
        column_info["Min"] = "-"
        column_info["Max"] = "-"

    audit_rows.append(column_info)

# Turn the audit results into a dataframe
final_audit = pd.DataFrame(audit_rows)

# Display all 40 columns
print(final_audit.to_string(index=False))

In [ ]:
# Allow pandas to display every row in the audit table
pd.set_option("display.max_rows", None)

# Show the complete final audit
print(final_audit.to_string(index=False))

In [ ]:
# Clean a copy of the dataset without changing the original
def clean_data(data):

    # Create a copy so the original raw dataset stays unchanged
    clean = data.copy(deep=True)

    # Correct the confirmed age error
    # Original source shows STU100001 should be age 17, not 170
    clean.loc[
        clean["Student_ID"] == "STU100001",
        "Age"
    ] = 17

    # Convert Enrollment_Date from text to a proper datetime data type
    clean["Enrollment_Date"] = pd.to_datetime(
        clean["Enrollment_Date"],
        dayfirst=True
    )

    # Remove exact duplicate records
    clean = clean.drop_duplicates()

    # Reset the row numbers after removing duplicates
    clean = clean.reset_index(drop=True)

    return clean

In [ ]:
# Create the cleaned version of the dataset
df_clean_preview = clean_data(df_copy)

# Check its size
print(df_clean_preview.shape)

In [ ]:
# Check that the duplicate was removed
print(
    "Duplicate rows:",
    df_clean_preview.duplicated().sum()
)

# Check that Student_ID values are now unique
print(
    "Duplicate Student_IDs:",
    df_clean_preview["Student_ID"].duplicated().sum()
)

# Check the corrected age
print(
    df_clean_preview.loc[
        df_clean_preview["Student_ID"] == "STU100001",
        ["Student_ID", "Age"]
    ]
)

# Check that Enrollment_Date is now a datetime data type
print(
    "Enrollment_Date type:",
    df_clean_preview["Enrollment_Date"].dtype
)

In [ ]:
# Final verification of the cleaned dataset

# Check for missing values
print(
    "Total missing values:",
    df_clean_preview.isna().sum().sum()
)

# Check age range after cleaning
print(
    "Age range:",
    df_clean_preview["Age"].min(),
    "to",
    df_clean_preview["Age"].max()
)

# Check for any negative numerical values
numerical_cols = df_clean_preview.select_dtypes(include="number").columns

negative_total = (
    df_clean_preview[numerical_cols] < 0
).sum().sum()

print(
    "Total negative numerical values:",
    negative_total
)

# Check final dataset size
print(
    "Final dataset shape:",
    df_clean_preview.shape
)

The confirmed age error for STU100001 was corrected from 170 to 17 based on the original source dataset. One exact duplicate record was removed, and Enrollment_Date was converted from text to a datetime data type. No missing values or negative numerical values were found, so no imputation or additional numerical cleaning was required. Unusual but source-consistent patterns, including assignment totals of 9 and scholarship payment records, were retained rather than altered without evidence.

In [ ]:
# Find all numerical columns
numerical_cols = df_copy.select_dtypes(include="number").columns

# Store the outlier results
outlier_summary = []

for col in numerical_cols:

    # Calculate the first and third quartiles
    q1 = df_copy[col].quantile(0.25)
    q3 = df_copy[col].quantile(0.75)

    # Calculate the interquartile range
    iqr = q3 - q1

    # Define the usual IQR outlier limits
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    # Count values outside the limits
    outlier_count = (
        (df_copy[col] < lower_bound)
        | (df_copy[col] > upper_bound)
    ).sum()

    # Save the result
    outlier_summary.append({
        "Column": col,
        "Lower_Bound": round(lower_bound, 2),
        "Upper_Bound": round(upper_bound, 2),
        "Outlier_Count": outlier_count
    })

# Convert the results into a dataframe
outlier_summary = pd.DataFrame(outlier_summary)

# Show only columns where potential outliers were found
print(
    outlier_summary[
        outlier_summary["Outlier_Count"] > 0
    ].to_string(index=False)
)

An IQR-based outlier check identified statistically unusual values in several numerical variables. These values were reviewed against the valid ranges established during data inspection. Most flagged values represented plausible student behaviour and were therefore retained. The age value of 170 was the only confirmed invalid outlier and was corrected to 17 using the original source dataset. No additional outliers were removed, capped, or transformed because there was insufficient evidence that they were data errors.

In [ ]:
# Find all CSV files in the data folder
# This helps us identify which file contains the full 100,000 records

from pathlib import Path

for file in Path("../data").glob("*.csv"):

    # Temporarily load each CSV
    temp_df = pd.read_csv(file)

    # Show the file name and dataset size
    print(file.name, temp_df.shape)

In [ ]:
# Load the full 100,000-row dataset
df_full = pd.read_csv(
    "../data/student-course-completion-prediction-dataset-original.csv"
)

# Use the first 30,000 records only
# We exclude the extra duplicate row from the trimmed file
df_30k = pd.read_csv(
    "../data/student-course-completion-prediction-dataset-trimmed.csv"
).iloc[:30000].copy()

print("Full dataset:", df_full.shape)
print("30k subset:", df_30k.shape)

In [ ]:
# Compare the percentage distribution of key categorical columns

compare_cols = [
    "Completed",
    "Gender",
    "Course_ID",
    "Payment_Mode"
]

for col in compare_cols:

    print(f"\n--- {col} ---")

    # Percentage distribution in the full 100k dataset
    full_pct = (
        df_full[col]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

    # Percentage distribution in the first 30k records
    subset_pct = (
        df_30k[col]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

    comparison = pd.DataFrame({
        "Full_100k_%": full_pct,
        "First_30k_%": subset_pct
    })

    print(comparison)

In [ ]:
# Compare key numerical variables between the full dataset
# and the first 30,000-record subset

numeric_compare_cols = [
    "Age",
    "Progress_Percentage",
    "Video_Completion_Rate",
    "Satisfaction_Rating",
    "Time_Spent_Hours"
]

comparison_rows = []

for col in numeric_compare_cols:

    comparison_rows.append({
        "Column": col,
        "Full_100k_Mean": round(df_full[col].mean(), 2),
        "First_30k_Mean": round(df_30k[col].mean(), 2)
    })

numeric_comparison = pd.DataFrame(comparison_rows)

print(numeric_comparison.to_string(index=False))

The first 30,000 records were retained as a manageable subset of the original 100,000-record dataset. Although the subset was selected sequentially rather than through random sampling, its representativeness was assessed by comparing key categorical distributions and numerical averages with the full dataset. Completion status, gender, course and payment-mode distributions were broadly similar. Numerical measures including age, progress percentage, video completion rate, satisfaction rating and time spent were also nearly identical. Therefore, the 30,000-record subset was considered sufficiently representative for the project analysis.

In [ ]:
# Print every column name vertically
# This makes all 40 columns easy to read

for i, col in enumerate(df_copy.columns, start=1):
    print(i, col)

In [ ]:
# Create descriptions for each variable in the dataset
# This documents what each of the 40 columns represents

variable_meanings = {
    "Student_ID": "Unique identifier assigned to each student",
    "Name": "Student name",
    "Gender": "Student gender",
    "Age": "Student age in years",
    "Education_Level": "Highest education level of the student",
    "Employment_Status": "Current employment status of the student",
    "City": "City where the student is located",
    "Device_Type": "Type of device used to access the course",
    "Internet_Connection_Quality": "Quality of the student's internet connection",
    "Course_ID": "Unique identifier for the course",
    "Course_Name": "Name of the enrolled course",
    "Category": "Subject category of the course",
    "Course_Level": "Difficulty or level of the course",
    "Course_Duration_Days": "Duration of the course in days",
    "Instructor_Rating": "Rating given to the course instructor",
    "Login_Frequency": "Frequency of student logins",
    "Average_Session_Duration_Min": "Average learning-session duration in minutes",
    "Video_Completion_Rate": "Percentage of course videos completed",
    "Discussion_Participation": "Level/count of participation in course discussions",
    "Time_Spent_Hours": "Amount of time spent on the course in hours",
    "Days_Since_Last_Login": "Number of days since the student's last login",
    "Notifications_Checked": "Number of course notifications checked",
    "Peer_Interaction_Score": "Score representing interaction with other students",
    "Assignments_Submitted": "Number of assignments submitted",
    "Assignments_Missed": "Number of assignments missed",
    "Quiz_Attempts": "Number of quiz attempts",
    "Quiz_Score_Avg": "Average quiz score",
    "Project_Grade": "Grade received for the course project",
    "Progress_Percentage": "Percentage of overall course progress",
    "Rewatch_Count": "Number of times course material was rewatched",
    "Payment_Mode": "Method used to pay for or access the course",
    "Fee_Paid": "Indicates whether the student paid the course fee",
    "Payment_Amount": "Recorded payment/course amount",
    "Discount_Used": "Indicates whether a discount was used",
    "App_Usage_Percentage": "Percentage representing use of the learning app",
    "Enrollment_Date": "Date the student enrolled in the course",
    "Reminder_Emails_Clicked": "Number of reminder emails clicked",
    "Support_Tickets_Raised": "Number of support requests raised",
    "Satisfaction_Rating": "Student satisfaction rating",
    "Completed": "Target variable showing whether the course was completed"
}

# Create a data dictionary containing:
# column name, data type and variable meaning
data_dictionary = pd.DataFrame({
    "Variable": df_copy.columns,
    "Data_Type": [str(df_copy[col].dtype) for col in df_copy.columns],
    "Meaning": [variable_meanings[col] for col in df_copy.columns]
})

# Display the complete data dictionary
print(data_dictionary.to_string(index=False))